Actividad de clase 

In [ ]:
import numpy as np
import tensorflow as tf
(x_train, y_train), (x_test, y_test)=tf.keras.datasets.mnist.load_data()
x_train=x_train.reshape(x_train.shape[0], 28*28)
x_test=x_test.reshape(x_test.shape[0], 28*28)
import matplotlib.pyplot as plt
plt.imshow(x_train[0].reshape(28,28), cmap='gray')

In [ ]:
In=tf.keras.layers.Input(shape=(784,))
Cp1=tf.keras.layers.Dense(64,activation='relu')(In)
Cp2=tf.keras.layers.Dense(64,activation='relu')(Cp1)
Out=tf.keras.layers.Dense(10,activation='softmax')(Cp2)
Model=tf.keras.Model(inputs=In,outputs=Out)

In [ ]:
Model.summary()

In [ ]:
Model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
Model.fit(x_train,y_train,epochs=30,batch_size=32)

## Parte 2 — Guardar el modelo y probarlo con imágenes propias

Hasta aquí el modelo aprendió con MNIST, pero vive solo en la memoria de este cuaderno. Ahora: (1) se re-entrena con píxeles normalizados a [0, 1], (2) se evalúa con datos que nunca vio, (3) se guarda en un archivo y (4) se usa desde una interfaz donde se arrastran imágenes.

In [ ]:
# Normalizar: de [0, 255] a [0, 1]. Las capas densas aprenden más estable con valores pequeños.
x_train_n = x_train.astype('float32') / 255.0
x_test_n = x_test.astype('float32') / 255.0

# Misma arquitectura, pesos nuevos
In = tf.keras.layers.Input(shape=(784,))
Cp1 = tf.keras.layers.Dense(64, activation='relu')(In)
Cp2 = tf.keras.layers.Dense(64, activation='relu')(Cp1)
Out = tf.keras.layers.Dense(10, activation='softmax')(Cp2)
Model = tf.keras.Model(inputs=In, outputs=Out)
Model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
Model.fit(x_train_n, y_train, epochs=15, batch_size=32, validation_split=0.1)

In [ ]:
# Evaluación con las 10 000 imágenes de prueba (el modelo nunca las vio)
perdida, exactitud = Model.evaluate(x_test_n, y_test, verbose=0)
print(f'Exactitud en prueba: {exactitud:.4f}')

# Guardar el modelo entrenado (arquitectura + pesos) en un archivo
Model.save('modelo_mnist.keras')

### Preprocesamiento: llevar una foto al formato MNIST
El archivo `preprocesamiento.py` (en la misma carpeta) quita la sombra del papel, invierte colores, recorta el trazo, lo escala a 20×20, lo centra en 28×28 y normaliza. Sin este paso el modelo se equivoca casi siempre con fotos reales.

In [ ]:
from PIL import Image
from preprocesamiento import preparar_imagen

img = Image.open('ejemplos/digito_3.png')   # cambia por tu propia foto
vector, vista = preparar_imagen(img)
probs = Model.predict(vector, verbose=0)[0]

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('Imagen original')
ax[1].imshow(vista, cmap='gray'); ax[1].set_title('Lo que ve el modelo (28x28)')
ax[2].bar(range(10), probs); ax[2].set_xticks(range(10)); ax[2].set_title(f'Predicción: {probs.argmax()}')
plt.show()

### Interfaz para arrastrar imágenes (Gradio)
Esta celda abre una mini página web dentro del cuaderno. Puedes soltar una imagen o dibujar el número.

In [ ]:
import app          # app.py carga 'modelo_mnist.keras' y define la interfaz
app.demo.launch()